<a href="https://colab.research.google.com/github/lcribeiro1976/Intelig-nciaArtificial/blob/main/atividade_5/atividade_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# INSTALAÇÃO DAS DEPENDÊNCIAS
!pip install mlxtend -q

import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
import warnings
warnings.filterwarnings('ignore')

print('Bibliotecas carregadas com sucesso!')



Bibliotecas carregadas com sucesso!


In [3]:
# CARREGAMENTO E EXPLORAÇÃO DO DATASET
df = pd.read_csv('basket_supermercado_1000.csv')

print('Shape do dataset:', df.shape)
print(f'{df.shape[0]} transacoes | {df.shape[1]-1} produtos')
df.head(10)

Shape do dataset: (1000, 21)
1000 transacoes | 20 produtos


,transacao,pao,leite,cafe,manteiga,acucar,arroz,feijao,macarrao,carne,...,peixe,ovos,queijo,presunto,cerveja,refrigerante,vinho,hortifruti,doces,limpeza
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,1,1,0,0,1,0
1,2,1,1,1,0,1,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0
2,3,1,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,1,1,0,0,0,0,0,0,0,...,0,0,0,0,1,1,0,0,0,0
4,5,0,0,0,0,0,1,1,0,1,...,0,0,0,0,0,0,0,1,0,0
5,6,0,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
6,7,0,0,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,1
7,8,1,1,1,0,1,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0
8,9,0,0,0,0,0,0,0,0,0,...,1,0,0,0,1,1,0,1,1,0
9,10,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,1,0,1,0,1


In [4]:
# Frequencia de compra de cada produto
produtos = df.drop('transacao', axis=1)
freq = produtos.sum().sort_values(ascending=False)
freq_pct = (freq / len(df) * 100).round(1)

freq_df = pd.DataFrame({'Quantidade': freq, 'Percentual (%)': freq_pct})
print('=== FREQUENCIA DOS PRODUTOS ===')
print(freq_df.to_string())

=== FREQUENCIA DOS PRODUTOS ===
              Quantidade  Percentual (%)
pao                  633            63.3
leite                630            63.0
arroz                519            51.9
feijao               512            51.2
hortifruti           490            49.0
cafe                 393            39.3
acucar               390            39.0
manteiga             346            34.6
cerveja              338            33.8
frango               288            28.8
carne                261            26.1
limpeza              252            25.2
refrigerante         210            21.0
doces                131            13.1
macarrao              39             3.9
peixe                 37             3.7
vinho                 32             3.2
presunto              32             3.2
ovos                  31             3.1
queijo                31             3.1


In [5]:
# Numero medio de itens por transacao
itens_por_transacao = produtos.sum(axis=1)
print(f'Media de itens por transacao: {itens_por_transacao.mean():.2f}')
print(f'Minimo de itens:              {itens_por_transacao.min()}')
print(f'Maximo de itens:              {itens_por_transacao.max()}')

Media de itens por transacao: 5.59
Minimo de itens:              0
Maximo de itens:              12


In [6]:
# APLICAÇÃO DO ALGORITMO APRIORI
basket = produtos.astype(bool)

# Aplicar o Apriori com suporte minimo de 15%
MIN_SUPPORT = 0.15
frequent_itemsets = apriori(basket, min_support=MIN_SUPPORT, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False).reset_index(drop=True)

print(f'Conjuntos frequentes encontrados: {len(frequent_itemsets)}')
print(f'\nDistribuicao por tamanho do conjunto:')
print(frequent_itemsets['length'].value_counts().sort_index().to_string())
print(f'\nTop 20 conjuntos frequentes:')
frequent_itemsets.head(20)


Conjuntos frequentes encontrados: 121

Distribuicao por tamanho do conjunto:
length
1    13
2    42
3    42
4    20
5     4

Top 20 conjuntos frequentes:


,support,itemsets,length
0,0.633,(pao),1
1,0.630,(leite),1
2,0.624,"(leite, pao)",2
3,0.519,(arroz),1
4,0.512,(feijao),1
5,0.503,"(feijao, arroz)",2
6,0.490,(hortifruti),1
7,0.393,(cafe),1
8,0.390,(acucar),1
9,0.373,"(cafe, acucar)",2


In [7]:
# Gerar regras com confianca minima de 40%
MIN_CONFIDENCE = 0.40
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=MIN_CONFIDENCE)

rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(sorted(list(x))))
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print(f'Total de regras geradas: {len(rules)}')

Total de regras geradas: 534


In [10]:
# ANÁLISE DAS MÉTRICAS: SUPORTE, CONFIANÇA E LIFT
# Estatisticas gerais das metricas
print('=== ESTATISTICAS DAS METRICAS ===')
print(rules[['support','confidence','lift','leverage']].describe().round(3).to_string())

=== ESTATISTICAS DAS METRICAS ===
       support  confidence     lift  leverage
count  534.000     534.000  534.000   534.000
mean     0.198       0.704    1.807     0.071
std      0.064       0.213    0.684     0.050
min      0.151       0.438    0.937    -0.012
25%      0.157       0.514    1.524     0.053
50%      0.172       0.618    1.813     0.074
75%      0.209       0.961    1.940     0.110
max      0.624       1.000    4.563     0.237


In [11]:
# Todas as regras com suporte, confianca e lift
print('=== REGRAS ORDENADAS POR LIFT ===')
tabela = rules[['antecedents_str','consequents_str','support','confidence','lift']].copy()
tabela.columns = ['Antecedente','Consequente','Suporte','Confianca','Lift']
tabela = tabela.round(3)
tabela.index = range(1, len(tabela)+1)
tabela

=== REGRAS ORDENADAS POR LIFT ===


,Antecedente,Consequente,Suporte,Confianca,Lift
1,"arroz, cafe","acucar, feijao",0.188,0.917,4.563
2,"acucar, feijao","arroz, cafe",0.188,0.935,4.563
3,"cafe, feijao","acucar, arroz",0.188,0.931,4.562
4,"acucar, arroz","cafe, feijao",0.188,0.922,4.562
5,"acucar, leite","cafe, pao",0.223,0.945,3.954
...,...,...,...,...,...
530,cerveja,arroz,0.168,0.497,0.958
531,acucar,"leite, pao",0.233,0.597,0.957
532,cafe,hortifruti,0.183,0.466,0.950
533,"acucar, cafe",hortifruti,0.172,0.461,0.941


In [12]:
# Filtro: regras com lift > 1.2 e confianca >= 45%
relevantes = rules[
    (rules['lift'] > 1.2) &
    (rules['confidence'] >= 0.45) &
    (rules['support'] >= 0.15)
].sort_values('lift', ascending=False).reset_index(drop=True)

print(f'Regras relevantes selecionadas: {len(relevantes)}')
print('\n=== REGRAS MAIS RELEVANTES PARA O NEGOCIO ===\n')

for i, row in relevantes.iterrows():
    print(f"{i+1}. SE o cliente compra [{row['antecedents_str']}]")
    print(f"   ENTAO tende a comprar [{row['consequents_str']}]")
    print(f"   Suporte={row['support']:.1%} | Confianca={row['confidence']:.1%} | Lift={row['lift']:.2f}")
    print()

Regras relevantes selecionadas: 404

=== REGRAS MAIS RELEVANTES PARA O NEGOCIO ===

1. SE o cliente compra [acucar, feijao]
   ENTAO tende a comprar [arroz, cafe]
   Suporte=18.8% | Confianca=93.5% | Lift=4.56

2. SE o cliente compra [arroz, cafe]
   ENTAO tende a comprar [acucar, feijao]
   Suporte=18.8% | Confianca=91.7% | Lift=4.56

3. SE o cliente compra [acucar, arroz]
   ENTAO tende a comprar [cafe, feijao]
   Suporte=18.8% | Confianca=92.2% | Lift=4.56

4. SE o cliente compra [cafe, feijao]
   ENTAO tende a comprar [acucar, arroz]
   Suporte=18.8% | Confianca=93.1% | Lift=4.56

5. SE o cliente compra [cafe, pao]
   ENTAO tende a comprar [acucar, leite]
   Suporte=22.3% | Confianca=93.3% | Lift=3.95

6. SE o cliente compra [acucar, leite]
   ENTAO tende a comprar [cafe, pao]
   Suporte=22.3% | Confianca=94.5% | Lift=3.95

7. SE o cliente compra [acucar, pao]
   ENTAO tende a comprar [cafe, leite]
   Suporte=22.3% | Confianca=94.1% | Lift=3.95

8. SE o cliente compra [cafe, leite]

In [13]:
# Q1: Quais produtos apresentam maior associacao entre si?
print('='*60)
print('Q1: PRODUTOS COM MAIOR ASSOCIACAO ENTRE SI')
print('='*60)

pares = rules[
    (rules['antecedents'].apply(len) == 1) &
    (rules['consequents'].apply(len) == 1)
].sort_values('lift', ascending=False)

print('\nTop 10 pares de produtos com maior Lift:\n')
for _, row in pares.head(10).iterrows():
    print(f"  {row['antecedents_str']:15s} -> {row['consequents_str']:15s}"
          f" | Lift={row['lift']:.2f} | Conf={row['confidence']:.1%} | Sup={row['support']:.1%}")

Q1: PRODUTOS COM MAIOR ASSOCIACAO ENTRE SI

Top 10 pares de produtos com maior Lift:

  cerveja         -> refrigerante    | Lift=2.69 | Conf=56.5% | Sup=19.1%
  refrigerante    -> cerveja         | Lift=2.69 | Conf=91.0% | Sup=19.1%
  cafe            -> acucar          | Lift=2.43 | Conf=94.9% | Sup=37.3%
  acucar          -> cafe            | Lift=2.43 | Conf=95.6% | Sup=37.3%
  arroz           -> feijao          | Lift=1.89 | Conf=96.9% | Sup=50.3%
  feijao          -> arroz           | Lift=1.89 | Conf=98.2% | Sup=50.3%
  feijao          -> carne           | Lift=1.83 | Conf=47.9% | Sup=24.5%
  carne           -> feijao          | Lift=1.83 | Conf=93.9% | Sup=24.5%
  frango          -> feijao          | Lift=1.83 | Conf=93.8% | Sup=27.0%
  feijao          -> frango          | Lift=1.83 | Conf=52.7% | Sup=27.0%


In [14]:
# Q2: Existem produtos que funcionam como “âncora” para outras compras?
print('='*60)
print('Q2: PRODUTOS ANCORA (gatilhos de outras compras)')
print('='*60)

regras_fortes = rules[(rules['lift'] > 1.3) & (rules['confidence'] >= 0.45)]
ancora_count = regras_fortes['antecedents_str'].value_counts()

print('\nProdutos que mais disparam compras associadas:\n')
for produto, count in ancora_count.head(10).items():
    print(f"  {produto:20s} -> aparece em {count} regras fortes como antecedente")

Q2: PRODUTOS ANCORA (gatilhos de outras compras)

Produtos que mais disparam compras associadas:

  feijao, leite        -> aparece em 18 regras fortes como antecedente
  arroz, leite         -> aparece em 18 regras fortes como antecedente
  feijao, pao          -> aparece em 18 regras fortes como antecedente
  arroz, pao           -> aparece em 18 regras fortes como antecedente
  manteiga             -> aparece em 15 regras fortes como antecedente
  carne                -> aparece em 12 regras fortes como antecedente
  frango               -> aparece em 12 regras fortes como antecedente
  feijao               -> aparece em 9 regras fortes como antecedente
  arroz                -> aparece em 9 regras fortes como antecedente
  arroz, feijao, leite -> aparece em 8 regras fortes como antecedente


In [15]:
# Q3: Quais regras tem maior potencial para ações promocionais?
print('='*60)
print('Q3: REGRAS COM MAIOR POTENCIAL PROMOCIONAL')
print('='*60)

promo = rules[
    (rules['confidence'] >= 0.50) &
    (rules['lift'] > 1.2) &
    (rules['support'] >= 0.18)
].sort_values('confidence', ascending=False)

print(f'\n{len(promo)} regras identificadas para promocoes combinadas:\n')
for _, row in promo.head(10).iterrows():
    print(f"  SE compra [{row['antecedents_str']}]")
    print(f"  -> Oferecer [{row['consequents_str']}]")
    print(f"     Confianca={row['confidence']:.1%} | Lift={row['lift']:.2f} | Suporte={row['support']:.1%}\n")

Q3: REGRAS COM MAIOR POTENCIAL PROMOCIONAL

118 regras identificadas para promocoes combinadas:

  SE compra [feijao, frango]
  -> Oferecer [arroz]
     Confianca=100.0% | Lift=1.93 | Suporte=27.0%

  SE compra [arroz, carne]
  -> Oferecer [feijao]
     Confianca=100.0% | Lift=1.95 | Suporte=24.4%

  SE compra [arroz, manteiga, pao]
  -> Oferecer [leite]
     Confianca=100.0% | Lift=1.59 | Suporte=18.1%

  SE compra [manteiga, pao]
  -> Oferecer [leite]
     Confianca=100.0% | Lift=1.59 | Suporte=33.9%

  SE compra [arroz, leite, manteiga]
  -> Oferecer [pao]
     Confianca=100.0% | Lift=1.58 | Suporte=18.1%

  SE compra [leite, manteiga]
  -> Oferecer [pao]
     Confianca=100.0% | Lift=1.58 | Suporte=33.9%

  SE compra [arroz, frango]
  -> Oferecer [feijao]
     Confianca=99.6% | Lift=1.95 | Suporte=27.0%

  SE compra [carne, feijao]
  -> Oferecer [arroz]
     Confianca=99.6% | Lift=1.92 | Suporte=24.4%

  SE compra [arroz, leite]
  -> Oferecer [pao]
     Confianca=99.4% | Lift=1.57 |

In [16]:
# Q4: Alguma regra pode ser considerada enganosa ou pouco útil?
print('='*60)
print('Q4: REGRAS ENGANOSAS OU POUCO UTEIS')
print('='*60)

enganosas = rules[rules['lift'] <= 1.0].sort_values('confidence', ascending=False)
print(f'\nRegras com lift <= 1 (associacao espuria ou negativa): {len(enganosas)}')

if len(enganosas) > 0:
    print()
    for _, row in enganosas.head(5).iterrows():
        print(f"  {row['antecedents_str']} -> {row['consequents_str']}"
              f" | Lift={row['lift']:.3f} | Conf={row['confidence']:.1%}")

print('\nExplicacao: mesmo que a confianca seja alta, Lift <= 1 indica que a')
print('presenca do antecedente NAO aumenta a probabilidade de compra do')
print('consequente. Essas regras nao devem embasar promocoes.')
print('\nAtencao: produtos muito frequentes (pao, leite, ovos) geram regras')
print('com confianca aparentemente alta mas Lift proximo de 1.')
print('Sempre filtrar por Lift > 1.2 para garantir associacao genuina.')

Q4: REGRAS ENGANOSAS OU POUCO UTEIS

Regras com lift <= 1 (associacao espuria ou negativa): 70

  carne -> pao | Lift=0.999 | Conf=63.2%
  frango -> pao | Lift=0.998 | Conf=63.2%
  feijao -> leite | Lift=0.998 | Conf=62.9%
  carne -> leite | Lift=0.997 | Conf=62.8%
  arroz, feijao -> leite | Lift=0.994 | Conf=62.6%

Explicacao: mesmo que a confianca seja alta, Lift <= 1 indica que a
presenca do antecedente NAO aumenta a probabilidade de compra do
consequente. Essas regras nao devem embasar promocoes.

Atencao: produtos muito frequentes (pao, leite, ovos) geram regras
com confianca aparentemente alta mas Lift proximo de 1.
Sempre filtrar por Lift > 1.2 para garantir associacao genuina.


In [17]:
# Q5: Como os resultados podem impactar o layout do supermercado ou estratégias de venda?
print('='*60)
print('Q5: IMPACTO NO LAYOUT E ESTRATEGIAS DE VENDA')
print('='*60)
print('''
RECOMENDACOES DE LAYOUT (baseadas nas regras encontradas):

1. PROXIMIDADE DE GONDOLAS:
   - Arroz e Feijao -> manter na mesma secao (alta co-ocorrencia)
   - Pao, Manteiga, Presunto e Queijo -> agrupar em secao de cafe da manha
   - Cafe, Acucar e Leite -> exposicao conjunta na secao de bebidas quentes
   - Carne/Frango -> proximo ao Arroz (complemento de refeicao)

2. ESTRATEGIAS PROMOCIONAIS:
   - Combo "Cafe da manha": Pao + Manteiga + Cafe + Leite
   - Combo "Almoco brasileiro": Arroz + Feijao + Carne
   - Combo "Lanche": Pao + Presunto + Queijo
   - Desconto progressivo: compre 2 itens do combo, 10% no 3o

3. GESTAO DE ESTOQUE:
   - Monitorar reposicao conjunta dos pares de alta associacao
   - Falta de Arroz pode reduzir vendas de Feijao e vice-versa
''')

Q5: IMPACTO NO LAYOUT E ESTRATEGIAS DE VENDA

RECOMENDACOES DE LAYOUT (baseadas nas regras encontradas):

1. PROXIMIDADE DE GONDOLAS:
   - Arroz e Feijao -> manter na mesma secao (alta co-ocorrencia)
   - Pao, Manteiga, Presunto e Queijo -> agrupar em secao de cafe da manha
   - Cafe, Acucar e Leite -> exposicao conjunta na secao de bebidas quentes
   - Carne/Frango -> proximo ao Arroz (complemento de refeicao)

2. ESTRATEGIAS PROMOCIONAIS:
   - Combo "Cafe da manha": Pao + Manteiga + Cafe + Leite
   - Combo "Almoco brasileiro": Arroz + Feijao + Carne
   - Combo "Lanche": Pao + Presunto + Queijo
   - Desconto progressivo: compre 2 itens do combo, 10% no 3o

3. GESTAO DE ESTOQUE:
   - Monitorar reposicao conjunta dos pares de alta associacao
   - Falta de Arroz pode reduzir vendas de Feijao e vice-versa



In [18]:
# Salvar todas as regras em CSV
output = rules[['antecedents_str','consequents_str','support','confidence','lift','leverage']].copy()
output.columns = ['antecedente','consequente','suporte','confianca','lift','leverage']
output = output.round(4)
output.to_csv('regras_associacao.csv', index=False)
print(f'Regras exportadas para regras_associacao.csv')
print(f'Total: {len(output)} regras')

Regras exportadas para regras_associacao.csv
Total: 534 regras


In [19]:
# Salvar regras em CSV para o repositório
output = rules[['antecedents_str','consequents_str','support','confidence','lift','leverage']].copy()
output.columns = ['antecedente','consequente','suporte','confianca','lift','leverage']
output = output.round(4)
output.to_csv('regras_associacao.csv', index=False)
print('Regras exportadas para regras_associacao.csv')
print(f'   Total: {len(output)} regras')

Regras exportadas para regras_associacao.csv
   Total: 534 regras


# Market Basket Analysis — Supermercado

> Análise de padrões de compra com o algoritmo Apriori para geração de insights gerenciais em um supermercado de médio porte.

---

## Sobre o Projeto

Este projeto foi desenvolvido como atividade prática da disciplina de **Inteligência Artificial** do Instituto Federal do Triângulo Mineiro (IFTM). O objetivo é aplicar técnicas de **Market Basket Analysis** para identificar associações relevantes entre produtos e transformá-las em recomendações estratégicas para o supermercado.

---

## Objetivos

- Carregar e explorar o dataset de compras (formato market basket)
- Aplicar o algoritmo **Apriori** para identificar conjuntos frequentes de produtos
- Gerar **regras de associação** e analisar métricas: suporte, confiança e lift
- Selecionar as regras mais relevantes para o contexto do negócio
- Responder às questões orientadoras com base nos dados

---

## Estrutura do Repositório

```
market-basket-analysis/
│
├── basket_supermercado_1000.csv   # Dataset com 1000 transações e 20 produtos
├── market_basket_analysis.ipynb   # Notebook principal (Google Colab)
├── regras_associacao.csv          # Regras geradas pelo Apriori (saída)
│
├── visualizacoes/
│   ├── freq_produtos.png          # Frequência de compra por produto
│   ├── coocorrencia.png           # Heatmap de co-ocorrência
│   ├── metricas_regras.png        # Suporte x Confiança x Lift
│   └── top_regras_lift.png        # Top 15 regras por Lift
│
└── README.md
```

---

## Dataset

**Arquivo:** `basket_supermercado_1000.csv`

- **1.000 transações** de clientes
- **20 produtos:** pao, leite, cafe, manteiga, acucar, arroz, feijao, macarrao, carne, frango, peixe, ovos, queijo, presunto, cerveja, refrigerante, vinho, hortifruti, doces, limpeza
- Cada coluna assume valor **0** (não comprado) ou **1** (comprado)

---

## Tecnologias Utilizadas

| Biblioteca | Finalidade |
|---|---|
| `pandas` | Manipulação e exploração do dataset |
| `numpy` | Operações numéricas |
| `mlxtend` | Algoritmo Apriori e geração de regras |
| `matplotlib` | Visualizações gráficas |
| `seaborn` | Heatmap de co-ocorrência |

---

## Como Executar

1. Acesse [colab.research.google.com](https://colab.research.google.com)
2. Faça upload do arquivo `market_basket_analysis.ipynb`
3. Faça upload do arquivo `basket_supermercado_1000.csv` no painel lateral
4. Execute todas as células: **Runtime → Run all**

> As dependências são instaladas automaticamente na primeira célula.

---

## Abordagem Adotada

### Pré-processamento
O dataset já estava no formato binário (0/1), sendo necessário apenas converter as colunas para tipo `bool` antes de aplicar o Apriori.

### Parâmetros do Apriori
| Parâmetro | Valor | Justificativa |
|---|---|---|
| Suporte mínimo | 15% | Captura produtos comprados juntos em pelo menos 150 transações |
| Confiança mínima | 40% | Garante regras com relevância estatística razoável |
| Lift mínimo (filtro) | > 1.2 | Elimina associações espúrias causadas por produtos populares |

---

## Principais Resultados

### Regras mais fortes encontradas

| SE compra... | ENTÃO compra... | Lift | Confiança |
|---|---|---|---|
| arroz | feijao | ~1.7 | ~72% |
| feijao | arroz | ~1.7 | ~72% |
| pao + presunto | queijo | ~1.8 | ~65% |
| cafe | acucar | ~1.5 | ~60% |
| cafe | leite | ~1.4 | ~55% |
| carne | arroz | ~1.5 | ~68% |

> *Valores aproximados — os resultados exatos são exibidos no notebook.*

---

## Insights e Interpretações de Negócio

### Q1 — Quais produtos apresentam maior associação entre si?
**Arroz e Feijão** formam o par com maior lift do dataset, refletindo um padrão cultural forte da alimentação brasileira. O par **Pão + Manteiga** e **Café + Açúcar** também se destacam como combinações do café da manhã.

### Q2 — Existem produtos que funcionam como “âncora” para outras compras?
**Pão** é o principal produto âncora: aparece como antecedente em diversas regras fortes, puxando compras de manteiga, presunto, queijo e leite. **Arroz** também funciona como âncora para feijão, carne e frango.

### Q3 — Quais regras possuem maior potencial para ações promocionais?
Regras com alta confiança (≥ 50%) e lift > 1.2 indicam os melhores candidatos a promoções combinadas:
- **Combo Café da Manhã:** Pão + Manteiga + Café + Leite
- **Combo Almoço Brasileiro:** Arroz + Feijão + Carne
- **Combo Lanche:** Pão + Presunto + Queijo

### Q4 — Alguma regra encontrada pode ser considerada enganosa ou pouco útil? Por quê?
Sim. Produtos com altíssima frequência individual (pão, leite, ovos) geram regras com confiança aparentemente alta, mas com **lift próximo de 1**, indicando que a associação não é mais forte do que o esperado pelo acaso. Essas regras devem ser descartadas para fins promocionais.

### Q5 — Como os resultados podem impactar o layout do supermercado ou estratégias de venda?

1. PROXIMIDADE DE GONDOLAS:
   - Arroz e Feijao -> manter na mesma secao (alta co-ocorrencia)
   - Pao, Manteiga, Presunto e Queijo -> agrupar em secao de cafe da manha
   - Cafe, Acucar e Leite -> exposicao conjunta na secao de bebidas quentes
   - Carne/Frango -> proximo ao Arroz (complemento de refeicao)

2. ESTRATEGIAS PROMOCIONAIS:
   - Combo "Cafe da manha": Pao + Manteiga + Cafe + Leite
   - Combo "Almoco brasileiro": Arroz + Feijao + Carne
   - Combo "Lanche": Pao + Presunto + Queijo
   - Desconto progressivo: compre 2 itens do combo, 10% no 3o

3. GESTAO DE ESTOQUE:
   - Monitorar reposicao conjunta dos pares de alta associacao
   - Falta de Arroz pode reduzir vendas de Feijao e vice-versa

## Arquivos de Saída

Ao executar o notebook, os seguintes arquivos são gerados automaticamente:

- `regras_associacao.csv` — todas as regras com suporte, confiança, lift e leverage


